# Demonstrating varieties of Query Classes

### Importing necessary classes and initiating Lucene VM

In [1]:
import lucene

# Initialize Lucene VM
lucene.initVM()

Mar 25, 2026 5:20:01 AM org.apache.lucene.internal.vectorization.PanamaVectorizationProvider <init>
INFO: Java vector incubator API enabled; uses preferredBitSize=512; FMA enabled


### Setting up general classes for search

In [3]:
from org.apache.lucene.index import DirectoryReader
from org.apache.lucene.store import FSDirectory

from org.apache.lucene.search import IndexSearcher
from java.io import File
import os

def setup_index(index_path):
    """Configure the index for searching"""
    if not os.path.exists(index_path):
        print("Searching index not exists.")
    else:
        indexDir = FSDirectory.open(File(index_path).toPath())
        searcher = IndexSearcher(DirectoryReader.open(indexDir))

    return searcher 

### A generic `search()` function that executes the actual search

In [4]:
def search(searcher, query, description, max_results=10):
    """Execute search and display results"""
    
    print(f"\n{description}")
    print("=" * 80)
    
    results = searcher.search(query, max_results)
    print(f"Total hits: {results.totalHits.value()}")
    
    for i, score_doc in enumerate(results.scoreDocs):
        doc = searcher.storedFields().document(score_doc.doc)
        print(f"{i+1}. Score: {score_doc.score:.4f}")
        print(f"   Title: {doc.get('TITLE')} ({doc.get('YEAR')})")
        print(f"   Director: {doc.get('DIRECTOR')}")
        print(f"   Genre: {doc.get('GENRE')}")
        print(f"   Plot: {doc.get('PLOT')}...")
        print()
    
    # reader.close()
    return results


### Initialize the IndexSearcher object

In [5]:
searcher = setup_index('/mnt/movie_index')


### Analyzing the individual terms

In [6]:
from org.apache.lucene.analysis.tokenattributes import CharTermAttribute
from java.io import StringReader
from org.apache.lucene.analysis.en import EnglishAnalyzer

analyzer = EnglishAnalyzer()

def get_stemmed_terms(analyzer, field, text):
    """Get stemmed terms from text using analyzer"""
    stemmed_terms = []
    stream = analyzer.tokenStream(field, StringReader(text))
    stream.reset()
    while stream.incrementToken():
        term_attr = stream.getAttribute(CharTermAttribute.class_)
        stemmed_terms.append(term_attr.toString())
    stream.close()
    return stemmed_terms


### TermQuery

In [8]:
from org.apache.lucene.analysis.en import EnglishAnalyzer
from org.apache.lucene.queryparser.classic import QueryParser
from org.apache.lucene.index import Term

from org.apache.lucene.search import TermQuery

def demonstrate_term_query(searcher):

    analyzer = EnglishAnalyzer()

    """TermQuery - exact term matching"""
    print("\n" + "=" * 80)
    print("TERM QUERY - Exact term matching")
    print("=" * 80)
    
    # Search for exact term "crime" in GENRE
    query1 = TermQuery(Term("GENRE", "Comedy"))
    print(query1)
    search(searcher, query1, "TermQuery: Genre contains 'Comedy'")
    
    # Search for specific origin
    # query2 = TermQuery(Term("ORIGIN", "Bollywood"))
    # print(query2)
    # search(searcher, query2, "TermQuery: Origin is 'Bollywood'", 3000)

    # # Search with general QueryParser
    # query3 = QueryParser("ORIGIN", analyzer).parse("Bollywood")
    # print(query3)
    # search(searcher, query3, "General analyzed Query: Origin is 'Bollywood'")
  

demonstrate_term_query(searcher)


TERM QUERY - Exact term matching
GENRE:Comedy

TermQuery: Genre contains 'Comedy'
Total hits: 0


### PhraseQuery

In [9]:
from org.apache.lucene.search import PhraseQuery

def demonstrate_phrase_query(searcher):
    """PhraseQuery - exact phrase matching"""
    print("\n" + "=" * 80)
    print("PHRASE QUERY - Exact phrase matching")
    print("=" * 80)
    
    # # Exact phrase in PLOT
    # builder1 = PhraseQuery.Builder()

    # for qterm in get_stemmed_terms(analyzer, "PLOT", "organized crime"):
    #     print(qterm)
    #     builder1.add(Term("PLOT", qterm))
    

    # # builder1.add(Term("PLOT", "organized"))
    # # builder1.add(Term("PLOT", "crime"))
    # query1 = builder1.build()
    # print(query1)
    # search(searcher, query1, "PhraseQuery: PLOT contains 'organized crime'")
    
    # # Phrase with slop (allowing some words between)
    # builder2 = PhraseQuery.Builder()
    # for qterm in get_stemmed_terms(analyzer, "PLOT", "psychological test"):
    #     print(qterm)
    #     builder2.add(Term("PLOT", qterm))
    # # builder2.add(Term("PLOT", "psychological"))
    # # builder2.add(Term("PLOT", "test"))
    # builder2.setSlop(3)  # Allow up to 3 words between
    # query2 = builder2.build()
    # print(query2)
    # search(searcher, query2, "PhraseQuery: PLOT contains 'psychological' within 3 words of 'test'")
    
    # Multi-word phrase
    builder3 = PhraseQuery.Builder()
    for qterm in get_stemmed_terms(analyzer, "PLOT", "dream sharing technology"):
        print(qterm)
        builder3.add(Term("PLOT", qterm))
    # builder3.add(Term("PLOT", "dream"))
    # builder3.add(Term("PLOT", "sharing"))
    # builder3.add(Term("PLOT", "technology"))
    builder3.setSlop(2)
    query3 = builder3.build()
    print(query3)
    search(searcher, query3, "PhraseQuery: PLOT contains 'dream sharing technology'")

demonstrate_phrase_query(searcher)



PHRASE QUERY - Exact phrase matching
dream
share
technolog
PLOT:"dream share technolog"~2

PhraseQuery: PLOT contains 'dream sharing technology'
Total hits: 1
1. Score: 2.9521
   Title: Inception (2010)
   Director: Christopher Nolan
   Genre: science fiction
   Plot: Dominick "Dom" Cobb and Arthur are "extractors", who perform corporate espionage using an experimental military technology to infiltrate the subconscious of their targets and extract valuable information through a shared dream world. Their latest target, Japanese businessman Saito, reveals that he arranged their mission himself to test Cobb for a seemingly impossible job: planting an idea in a person's subconscious, or "inception". To break up the energy conglomerate of ailing competitor Maurice Fischer, Saito wants Cobb to convince Fischer's son and heir, Robert, to dissolve his father's company. In return, Saito promises to use his influence to clear Cobb of a murder charge, allowing Cobb to return home to his children

### TermRangeQuery

In [11]:
from org.apache.lucene.search import TermRangeQuery

from org.apache.lucene.util import BytesRef

def demonstrate_term_range_query(searcher):
    """TermRangeQuery - range queries on text fields"""
    print("\n" + "=" * 80)
    print("TERM RANGE QUERY - Range queries on text fields")
    print("=" * 80)
        
    # Range of TITLEs (alphabetical)
    query2 = TermRangeQuery(
        "TITLE",
        BytesRef("m"),  # Titles starting from M
        BytesRef("t"),  # Titles up to T
        True,
        True
    )
    print(query2)
    search(searcher, query2, "TermRangeQuery: Titles from M to T", max_results=100)

demonstrate_term_range_query(searcher)



TERM RANGE QUERY - Range queries on text fields
TITLE:[m TO t]

TermRangeQuery: Titles from M to T
Total hits: 1001
1. Score: 1.0000
   Title: Kansas Saloon Smashers (1901)
   Director: Unknown
   Genre: unknown
   Plot: A bartender is working at a saloon, serving drinks to customers. After he fills a stereotypically Irish man's bucket with beer, Carrie Nation and her followers burst inside. They assault the Irish man, pulling his hat over his eyes and then dumping the beer over his head. The group then begin wrecking the bar, smashing the fixtures, mirrors, and breaking the cash register. The bartender then sprays seltzer water in Nation's face before a group of policemen appear and order everybody to leave.[1]...

2. Score: 1.0000
   Title: Love by the Light of the Moon (1901)
   Director: Unknown
   Genre: unknown
   Plot: The moon, painted with a smiling face hangs over a park at night. A young couple walking past a fence learn on a railing and look up. The moon smiles. They embra

### NumericRangeQuery

In [20]:
from org.apache.lucene.document import IntPoint

def demonstrate_numeric_range_query(searcher):
    """NumericRangeQuery - range queries on numeric fields"""
    print("\n" + "=" * 80)
    print("NUMERIC RANGE QUERY - Range queries on numeric fields")
    print("=" * 80)
    
    # Range of YEARs
    query = IntPoint.newRangeQuery("YEAR", 2010, 2012)
    search(searcher, query, "NumericRangeQuery: Movies from 1990 to 1999", 5000)
    
    # # Single YEAR
    # query = IntPoint.newExactQuery("YEAR", 1999)
    # search(searcher, query, "NumericRangeQuery: Movies from exactly 1994", 500)

demonstrate_numeric_range_query(searcher)



NUMERIC RANGE QUERY - Range queries on numeric fields

NumericRangeQuery: Movies from 1990 to 1999
Total hits: 2557
1. Score: 1.0000
   Title: 127 Hours (2010)
   Director: Danny Boyle
   Genre: biography, drama
   Plot: Mountaineer Aron Ralston goes hiking at Utah's Canyonlands National Park. He befriends hikers Kristi and Megan, and shows them an underground pool. After swimming, Aron parts ways with the hikers, and continues through a slot canyon in Blue John Canyon. While climbing down, he slips and falls, knocking a boulder which smashes his right hand and wrist against the wall. Stuck, he tries calling for help but realizes that he is alone.
Ralston begins recording a video diary to maintain morale, chipping away parts of the boulder in order to free himself and to keep warm at night. He rations his food and water, in order to survive the ordeal. He sets up a pulley using his climbing rope in a futile attempt to lift the boulder.
Days pass. Ralston considers using his pocket kni

### PrefixQuery

In [40]:
from org.apache.lucene.search import PrefixQuery

def demonstrate_prefix_query(searcher):
    """PrefixQuery - terms starting with specific prefix"""
    print("\n" + "=" * 80)
    print("PREFIX QUERY - Terms starting with specific prefix")
    print("=" * 80)
    
    # # Titles starting with "The"
    # query1 = PrefixQuery(Term("TITLE", "the"))
    # search(searcher, query1, "PrefixQuery: Titles starting with 'The'")
    
    # # Directors starting with "Chris"
    # query2 = PrefixQuery(Term("DIRECTOR", "chris"))
    # search(searcher, query2, "PrefixQuery: Directors starting with 'Chris'")
    
    # Genres starting with "Dra"
    query3 = PrefixQuery(Term("GENRE", "dra"))
    search(searcher, query3, "PrefixQuery: Genres starting with 'Dra'")

demonstrate_prefix_query(searcher)


PREFIX QUERY - Terms starting with specific prefix

PrefixQuery: Genres starting with 'Dra'
Total hits: 1001
1. Score: 1.0000
   Title: The Adventures of Dollie (1908)
   Director: D. W. Griffith
   Genre: drama
   Plot: On a beautiful summer day a father and mother take their daughter Dollie on an outing to the river. The mother refuses to buy a gypsy's wares. The gypsy tries to rob the mother, but the father drives him off. The gypsy returns to the camp and devises a plan. They return and kidnap Dollie while her parents are distracted. A rescue crew is organized, but the gypsy takes Dollie to his camp. They gag Dollie and hide her in a barrel before the rescue party gets to the camp. Once they leave the gypsies and escapes in their wagon. As the wagon crosses the river, the barrel falls into the water. Still sealed in the barrel, Dollie is swept downstream in dangerous currents. A boy who is fishing in the river finds the barrel, and Dollie is reunited safely with her parents....

2

### BooleanQuery

In [ ]:
from org.apache.lucene.search import BooleanQuery, BooleanClause
from org.apache.lucene.document import IntPoint


def demonstrate_boolean_query(searcher):
    """BooleanQuery - combining multiple queries"""
    print("\n" + "=" * 80)
    print("BOOLEAN QUERY - Combining multiple queries")
    print("=" * 80)
    
    # Crime movies from the 1990
    # builder = BooleanQuery.Builder()
    # builder.add(TermQuery(Term("GENRE", "crime")), BooleanClause.Occur.MUST)
    # builder.add(IntPoint.newRangeQuery("YEAR", 1990, 1999), BooleanClause.Occur.MUST)
    # query = builder.build()
    # print(query)
    # search(searcher, query, "BooleanQuery: Crime movies from 1990s")



    # # Drama movies but NOT from 1994
    # builder = BooleanQuery.Builder()
    # builder.add(TermQuery(Term("GENRE", "drama")), BooleanClause.Occur.MUST)
    # builder.add(IntPoint.newExactQuery("YEAR", 1994), BooleanClause.Occur.MUST_NOT)
    # query = builder.build()
    # print(query)
    # search(searcher, query, "BooleanQuery: Drama movies NOT from 1994")
    


    # Complex: (Crime OR Drama) AND (after 2000)
    builder = BooleanQuery.Builder()
    
    # OR subquery for GENREs
    # GENRE_builder = BooleanQuery.Builder()
    # GENRE_builder.add(TermQuery(Term("GENRE", "crime")), BooleanClause.Occur.SHOULD)
    # GENRE_builder.add(TermQuery(Term("GENRE", "drama")), BooleanClause.Occur.SHOULD)
    
    # builder.add(GENRE_builder.build(), BooleanClause.Occur.MUST)
    # builder.add(IntPoint.newRangeQuery("YEAR", 2000, 2020), BooleanClause.Occur.MUST)
    # query = builder.build()
    # print(query)
    # search(searcher, query, "BooleanQuery: (Crime OR Drama) AND after 2000")



    # Christopher Nolan OR Martin Scorsese
    # (Christopher AND Nolan) OR (Martin AND Scorsese)
    builder = BooleanQuery.Builder()
    # builder.add(TermQuery(Term("DIRECTOR", "christopher nolan")), BooleanClause.Occur.SHOULD)
    
    b1 = BooleanQuery.Builder()
    b1.add(TermQuery(Term("DIRECTOR", "christopher")), BooleanClause.Occur.MUST)
    b1.add(TermQuery(Term("DIRECTOR", "nolan")), BooleanClause.Occur.MUST)
    # builder.add(TermQuery(Term("DIRECTOR", "martin scorsese")), BooleanClause.Occur.SHOULD)
    query = builder.build()
    print(query)
    search(searcher, query, "BooleanQuery: Directed by Nolan OR Scorsese")
    

demonstrate_boolean_query(searcher)



BOOLEAN QUERY - Combining multiple queries
DIRECTOR:christopher nolan DIRECTOR:martin scorsese

BooleanQuery: Directed by Nolan OR Scorsese
Total hits: 0


### WildcardQuery

In [ ]:
from org.apache.lucene.search import WildcardQuery

def demonstrate_wildcard_query(searcher):
    """WildcardQuery - pattern matching with wildcards"""
    print("\n" + "=" * 80)
    print("WILDCARD QUERY - Pattern matching with wildcards")
    print("=" * 80)
    
    # Titles with "the" anywhere
    query1 = WildcardQuery(Term("TITLE", "*the*"))
    search(searcher, query1, "WildcardQuery: Titles containing 'the'")
    
    # Directors with last name starting with "N"
    query2 = WildcardQuery(Term("DIRECTOR", "* n*"))
    search(searcher, query2, "WildcardQuery: Directors with last name starting with 'N'")
    
    # Movies with "fight" in TITLE or PLOT
    builder3 = BooleanQuery.Builder()
    builder3.add(WildcardQuery(Term("TITLE", "*fight*")), BooleanClause.Occur.SHOULD)
    builder3.add(WildcardQuery(Term("PLOT", "*fight*")), BooleanClause.Occur.SHOULD)
    query3 = builder3.build()
    search(searcher, query3, "WildcardQuery: Contains 'fight' in TITLE OR PLOT")
    
    # Single character wildcard
    query4 = WildcardQuery(Term("TITLE", "f?ght*"))
    search(searcher, query4, "WildcardQuery: Titles matching 'f?ght*' pattern")

demonstrate_wildcard_query(searcher)

### FuzzyQuery

In [ ]:
from org.apache.lucene.search import FuzzyQuery

def demonstrate_fuzzy_query(searcher):
    """FuzzyQuery - approximate string matching"""
    print("\n" + "=" * 80)
    print("FUZZY QUERY - Approximate string matching")
    print("=" * 80)
    
    # Fuzzy search for DIRECTOR names
    query1 = FuzzyQuery(Term("DIRECTOR", "christophar"))  # Misspelling of Christopher
    search(searcher, query1, "FuzzyQuery: Director 'Christofer' (finds Christopher)")
    
    # Fuzzy search for GENREs
    query2 = FuzzyQuery(Term("GENRE", "draama"))  # Intentional misspelling
    search(searcher, query2, "FuzzyQuery: Genre 'drama' (finds Drama)")
    
    # Fuzzy search with custom maxEdits
    stemmed_terms = get_stemmed_terms(analyzer, "TITLE", "Godfater")
    query3 = FuzzyQuery(Term("TITLE", stemmed_terms[0]), 1)  # maxEdits=1
    search(searcher, query3, "FuzzyQuery: Title 'godfater' with maxEdits=1")
    
    # Fuzzy search in PLOT
    query4 = FuzzyQuery(Term("PLOT", "psycological"))  # Misspelling of psychological
    search(searcher, query4, "FuzzyQuery: PLOT contains 'psycological'")

demonstrate_fuzzy_query(searcher)

### MatchAllDocsQuery

In [ ]:
from org.apache.lucene.search import MatchAllDocsQuery

def demonstrate_match_all_query(searcher):
    """MatchAllDocsQuery - match all documents"""
    print("\n" + "=" * 80)
    print("MATCH ALL DOCS QUERY - Match all documents")
    print("=" * 80)
    
    # Get total number of documents to retrieve all
    total_docs = searcher.getIndexReader().numDocs()
    print(total_docs)

    # Match all documents
    query1 = MatchAllDocsQuery()
    search(searcher, query1, "MatchAllDocsQuery: All movies", 10)
    
    # Match all with filtering
    builder = BooleanQuery.Builder()
    builder.add(MatchAllDocsQuery(), BooleanClause.Occur.MUST)
    builder.add(IntPoint.newRangeQuery("YEAR", 2000, 2010), BooleanClause.Occur.FILTER)
    query2 = builder.build()
    search(searcher, query2, "MatchAllDocsQuery filtered by YEARs 2000-2010")

demonstrate_match_all_query(searcher)


In [ ]:
def run_all_examples(searcher):
    """Execute all query examples"""
    print("Wikipedia Movie PLOTs - Lucene Query Examples")
    print("=============================================")
    # indexPath = '/mnt/movie_index'
    
    # create_sample_movies_index()
    
    demonstrate_term_query(searcher)
    demonstrate_phrase_query(searcher)
    demonstrate_term_range_query(searcher)
    demonstrate_numeric_range_query(searcher)
    demonstrate_prefix_query(searcher)
    demonstrate_boolean_query(searcher)
    demonstrate_wildcard_query(searcher)
    demonstrate_fuzzy_query(searcher)
    demonstrate_match_all_query(searcher)
    demonstrate_combined_example(searcher)

# Run the complete example
searcher = setup_index('/mnt/movie_index')
run_all_examples(searcher)
# demonstrate_term_query(searcher)

